In [ ]:
# Name: Jackson Irungu Maina
# Reg No: ST61/55297/2025
# Programme: ST61 - Master of Data Science
# School: School of Science and Technology
# Course: Data Mining and Big Data , CSA 806 (Year 1 Sem 2 )
# Task: Module 1: Assignment 1

In [4]:
import time
from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
import pandas as pd

In [5]:
# Load the dataset
df = pd.read_csv('fiia.csv')

# Inspect the first few rows and column info
print(df.head())
print(df.info())
print(df['country'].unique())

  country  year    uniqueid bank_account location_type cellphone_access  \
0   Kenya  2018  uniqueid_1          Yes         Rural              Yes   
1   Kenya  2018  uniqueid_2           No         Rural               No   
2   Kenya  2018  uniqueid_3          Yes         Urban              Yes   
3   Kenya  2018  uniqueid_4           No         Rural              Yes   
4   Kenya  2018  uniqueid_5           No         Urban               No   

   household_size  age_of_respondent gender_of_respondent  \
0               3                 24               Female   
1               5                 70               Female   
2               5                 26                 Male   
3               5                 34               Female   
4               8                 26                 Male   

  relationship_with_head           marital_status  \
0                 Spouse  Married/Living together   
1      Head of Household                  Widowed   
2         Other relativ

In [6]:
# Filter for Kenya
kenya_df = df[df['country'] == 'Kenya'].copy()

# Features for clustering
features = ['household_size', 'age_of_respondent', 'location_type', 'cellphone_access', 'gender_of_respondent']

# Preprocessing
kenya_df['location_type'] = kenya_df['location_type'].map({'Rural': 0, 'Urban': 1})
kenya_df['cellphone_access'] = kenya_df['cellphone_access'].map({'No': 0, 'Yes': 1})
kenya_df['gender_of_respondent'] = kenya_df['gender_of_respondent'].map({'Female': 0, 'Male': 1})

X = kenya_df[features].dropna()

# Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [7]:
# Sample for speed if dataset is large (it's 6k rows for Kenya usually, checking)
print(f"Kenya sample size: {len(X_scaled)}")

Kenya sample size: 6068


In [8]:
# K-Means
start_time = time.time()
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(X_scaled)
kmeans_time = time.time() - start_time
kmeans_silhouette = silhouette_score(X_scaled, kmeans_labels)

In [9]:
# DBSCAN
start_time = time.time()
dbscan = DBSCAN(eps=0.5, min_samples=5)
dbscan_labels = dbscan.fit_predict(X_scaled)
dbscan_time = time.time() - start_time

In [10]:
# Silhouette score only works if more than one cluster is found (excluding noise)
n_clusters_dbscan = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
if n_clusters_dbscan > 1:
    dbscan_silhouette = silhouette_score(X_scaled, dbscan_labels)
else:
    dbscan_silhouette = "N/A (too few clusters)"

print(f"K-Means Time: {kmeans_time:.4f}s, Silhouette: {kmeans_silhouette:.4f}")
print(f"DBSCAN Time: {dbscan_time:.4f}s, Silhouette: {dbscan_silhouette}, Clusters: {n_clusters_dbscan}")

K-Means Time: 0.0478s, Silhouette: 0.2815
DBSCAN Time: 0.1375s, Silhouette: 0.35376843528879076, Clusters: 11
